In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 17:32:46.327419: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 17:32:49.581675: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 1,
        "ips": ['172.190.116.144'],
        "ports": [50151]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 1,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [ ]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 17:32:56,640 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 17:32:56,642 [DEBUG] [Rain] Rain is initialized
2023-07-05 17:32:56,645 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 17:32:56,647 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 17:32:56,650 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 17:32:56,652 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-05 17:32:56,654 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 17:32:56,656 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 17:32:56,658 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 17:32:56,784 [INFO] [Provisioner] provisioner is serving
2023-07-05 17:32:56,786 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 17:32:56,790 [INFO] [Coordinator] coordinator is serving
2023-07-05 17:32:56,792 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 17:32:56,871 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 17:32:56,877 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 17:32:56,883 [DEBUG] [LocalProvisioner] Creating 1 workers
2023-07-05 17:32:56,887 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 17:32:56,893 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 17:32:56,898 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
2023-07-05 17:32:56,905 [DEBUG] [DividerAmbassador] divider ambassador is serving


469/469 [==============================] - 4s 7ms/step - loss: 0.4218 - accuracy: 0.8706


2023-07-05 17:33:27,553 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 17:33:27,555 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-05 17:33:27,765 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 17:33:27,784 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-05 17:33:27,828 [DEBUG] [DeepLearning] Iteration 1/1 complete for worker 1.
2023-07-05 17:33:27,832 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-05 17:33:27,834 [DEBUG] [Divider] Divider stopped serving
2023-07-05 17:33:27,837 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 17:33:27,839 [DEBUG] [LocalProvisioner] Workers are deleted
2023-07-05 17:33:27,842 [INFO] [Provisioner] provisioner stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 4ms/step - loss: 0.1451 - accuracy: 0.9541

Test accuracy: 95.4%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 17:33:28,563 [INFO] [Provisioner] provisioner is serving
2023-07-05 17:33:28,565 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 17:33:28,568 [INFO] [Coordinator] coordinator is serving
2023-07-05 17:33:28,570 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 17:33:28,573 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-05 17:33:28,575 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 17:33:28,576 [DEBUG] [LocalProvisioner] Creating 1 workers
2023-07-05 17:33:28,578 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 17:33:28,580 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 17:33:28,580 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 17:33:28,583 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
202

469/469 [==============================] - 4s 7ms/step - loss: 0.1952 - accuracy: 0.9420
sending data to divider


2023-07-05 17:34:02,654 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 17:34:02,758 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-05 17:34:03,111 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 17:34:03,154 [DEBUG] [DeepLearning] Iteration 1/1 complete.
DEBUG:DeepLearning:Iteration 1/1 complete.
2023-07-05 17:34:03,165 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-05 17:34:03,167 [DEBUG] [Div

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.1057 - accuracy: 0.9665

Test accuracy: 96.6%
